## Popularity & Weighted Rating

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")

PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models/popularity"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Ready!")

Ready!


In [2]:
df = pd.read_csv(f"{PROCESSED_DIR}/popularity.csv")

print(f"Loaded: {len(df)} movies")
print(f"\nTop 5 by weighted score:")
df[["title","vote_average","vote_count","weighted_score"]].head()

Loaded: 9988 movies

Top 5 by weighted score:


,title,vote_average,vote_count,weighted_score
0,The Shawshank Redemption,8.719,30258,8.607825
1,The Godfather,8.700,22848,8.556630
2,The Dark Knight,8.500,35631,8.414918
3,Schindler's List,8.568,17391,8.395597
4,Interstellar,8.471,39573,8.395250


In [3]:
def get_top_movies(n: int = 10, genre: str = None, 
                   min_year: int = None, max_year: int = None) -> pd.DataFrame:
    """
    أفضل الأفلام بالـ Weighted Rating
    genre    : فلتر النوع (Action, Drama, ...)
    min_year : من سنة
    max_year : لسنة
    """
    result = df.copy()
    
    # فلتر النوع
    if genre:
        result = result[result["genres"].str.contains(genre, case=False, na=False)]
    
    # فلتر السنة
    if min_year:
        result = result[pd.to_numeric(result["release_year"], errors="coerce") >= min_year]
    if max_year:
        result = result[pd.to_numeric(result["release_year"], errors="coerce") <= max_year]
    
    return result.head(n)[["movie_id","title","genres","release_year",
                            "vote_average","vote_count","weighted_score","poster_path"]]


def get_trending(n: int = 10, genre: str = None) -> pd.DataFrame:
    """الأفلام الأكثر شهرة (بالـ popularity score)"""
    result = df.copy()
    
    if genre:
        result = result[result["genres"].str.contains(genre, case=False, na=False)]
    
    result = result.sort_values("popularity", ascending=False)
    return result.head(n)[["movie_id","title","genres","release_year",
                            "vote_average","popularity","poster_path"]]


def get_hidden_gems(n: int = 10, min_score: float = 7.5, 
                    max_votes: int = 1000) -> pd.DataFrame:
    """أفلام تقييمها عالي لكن مش مشهورة"""
    result = df[
        (df["vote_average"] >= min_score) &
        (df["vote_count"] <= max_votes) &
        (df["vote_count"] >= 100)
    ].sort_values("vote_average", ascending=False)
    
    return result.head(n)[["movie_id","title","genres","release_year",
                            "vote_average","vote_count","poster_path"]]

print("All functions ready!")

All functions ready!


In [4]:
print("=== Top 10 Movies (All Time) ===")
print(get_top_movies(10)[["title","vote_average","weighted_score"]].to_string())

=== Top 10 Movies (All Time) ===
                                           title  vote_average  weighted_score
0                       The Shawshank Redemption         8.719        8.607825
1                                  The Godfather         8.700        8.556630
2                                The Dark Knight         8.500        8.414918
3                               Schindler's List         8.568        8.395597
4                                   Interstellar         8.471        8.395250
5                                   Pulp Fiction         8.484        8.384987
6  The Lord of the Rings: The Return of the King         8.497        8.384243
7                                  Spirited Away         8.534        8.371831
8                                   Forrest Gump         8.464        8.364659
9                          The Godfather Part II         8.571        8.359131


In [5]:
print("=== Top Action Movies ===")
print(get_top_movies(10, genre="Action")[["title","vote_average","weighted_score"]].to_string())

print("\n=== Top Drama Movies ===")
print(get_top_movies(10, genre="Drama")[["title","vote_average","weighted_score"]].to_string())

=== Top Action Movies ===
                                                title  vote_average  weighted_score
2                                     The Dark Knight         8.500        8.414918
6       The Lord of the Rings: The Return of the King         8.497        8.384243
13  The Lord of the Rings: The Fellowship of the Ring         8.433        8.327911
14                                          Inception         8.372        8.299602
15              The Lord of the Rings: The Two Towers         8.416        8.297042
20                            The Empire Strikes Back         8.393        8.243727
22                  Spider-Man: Into the Spider-Verse         8.393        8.234303
29                             Avengers: Infinity War         8.234        8.152967
31                                         The Matrix         8.243        8.150241
32                                  Avengers: Endgame         8.234        8.141276

=== Top Drama Movies ===
                       t

In [6]:
print("=== Trending Now ===")
print(get_trending(10)[["title","genres","popularity"]].to_string())

print("\n=== Hidden Gems (High Rating, Low Votes) ===")
print(get_hidden_gems(10)[["title","genres","vote_average","vote_count"]].to_string())

=== Trending Now ===
                             title                                             genres  popularity
4431  The Super Mario Galaxy Movie          Family|Comedy|Adventure|Fantasy|Animation    443.9285
3506                       Swapped                 Adventure|Animation|Family|Fantasy    409.3611
6759                          Apex                                    Action|Thriller    369.7562
980                         Malena                                              Drama    310.2717
2609                     Send Help                             Horror|Thriller|Comedy    247.9714
562          The Devil Wears Prada                                       Drama|Comedy    237.6777
4777     Your Heart Will Be Broken                                      Romance|Drama    223.8870
1951                       Michael                                        Music|Drama    197.8728
4550       The Devil Wears Prada 2                                       Comedy|Drama    173.3912

In [7]:
print("Saving...")
df.to_csv(f"{MODELS_DIR}/popularity_df.csv", index=False)
joblib.dump({
    "C": df["vote_average"].mean(),
    "m": df["vote_count"].quantile(0.70)
}, f"{MODELS_DIR}/rating_params.pkl")

print("Done!")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(f"{MODELS_DIR}/{f}") / 1024
    print(f"  {f:35s} {size:.1f} KB")

Saving...
Done!
  popularity_df.csv                   4110.0 KB
  rating_params.pkl                   0.1 KB
